In [ ]:
import json
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.utils import ANALYSIS_TYPES, get_countries_by_continent, split_list_into_segments, get_analysis_paths, get_analysis_commits_df
from emu_renewal.plotting import plot_param_post_comparison, get_flat_priors

set_matplotlib_formats("svg")

# Purpose
This document presents the posterior distributions of the two parameters that
define the floored scaling family of analyses used for the floored OxCGRT
and mobility approaches: the scaling floor $f$ and the scaling exponent $m$.
In these analysis approaches:
$$
M_t = [f + (1-f)\,S_t]^m,
$$
where $S_t$ is the weighted policy or complement-of-restriction signal
(one when unrestricted or at its reference level, zero at maximum restriction
or zero mobility).
The floor parameter $f$ is the residual level of the population interaction proxy
under the strongest possible intervention measures: 
even with policies at their most restrictive 
(or mobility at zero), the signal cannot be reduced below $f$, 
such a fraction $f$ of unrestricted contact remains before the effect of
the exponent parameter is applied.
The exponent parameter $m$ then governs how strongly reductions in 
the signal translate into reductions in transmission 
($m=1$ leaves the floored signal unchanged; 
$m>1$ deepens reductions as the signal drops further below one).

Priors on these parameters were both set to be uniform over plausible bounds
($f\sim\mathrm{Beta}(1,1)$ and $m\sim\mathrm{Uniform}(0,2)$).
Because $f$ and $m$ trade off, their marginal posteriors are not fully
separable: similar realised $M_t$ could arise from a higher floor with a larger
exponent or a lower floor with a smaller exponent.
We therefore take our primary overall metric to be $(1-f)^m$,
results from which are presented in the following document.
Note that the independent OxCGRT analysis is omitted, because it does not use $f$ or $m$.

Applying a floor was necessary because the OxCGRT metrics could reach 
full restrictions across all domains, implying $S_t=0$,
which did not support epidemiological fitting to data.
To match this approach, our analyses considering the effects of mobility 
were adapted from the approach in our previous paper to align structurally 
with this policy-focused approach.

In countries for which effects of policies and mobility are clear,
mobility floors often sit lower than policy floors.
This matches our intuition that zero mobility could conceptually come close to
halting transmission through reduced community interactions,
whereas maximum recorded policy intensity reaches a ceiling effect prior to this point.

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json", "r"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
countries_by_cont = get_countries_by_continent(analysis_paths)
analyses = [a for a in ANALYSIS_TYPES if a != "oxcgrt_independent"]

prior_info = get_flat_priors()
for param in ["scale_floor", "scale_exp"]:
    display(Markdown(f"# {prior_info[param]['short_name'].capitalize()} parameter"))
    for cont, cont_countries in countries_by_cont.items():
        cont_name = pc.convert_continent_code_to_continent_name(cont)
        display(Markdown(f"## {cont_name}"))
        for countries in split_list_into_segments(cont_countries, 16):
            display(plot_param_post_comparison(countries, analysis_paths, param))

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())